# Semantique with Dask:
# Sentinel-2 Scene Classification Layer PoC

Simple cloud & snow time-series and filtering. Based on other semantique demo recipes.

In [1]:
import json
import logging
import warnings

import dask.distributed as dd
import geopandas as gpd
from pystac_client import Client
import shapely

# For visualization
import holoviews as hv
import hvplot.xarray

hv.extension("bokeh")

In [2]:
import semantique as sq

/jupyter/semantique/semantique/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# Configure logging
logging.basicConfig(
    level=logging.WARNING,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True  # This forces reconfiguration if already set
)

logging.getLogger("semantique").setLevel(logging.DEBUG)

In [4]:
# Start Dask LocalCluster and dashboard
# By default, dashboard will be available at http://localhost:8787
dask_client = dd.Client(n_workers=6, threads_per_worker=1)
print(dask_client.dashboard_link)

http://127.0.0.1:8787/status


In [5]:
# Define area of interest and time period

# Stockerau, Austria (small)
aoi = shapely.from_wkt("POLYGON ((16.17 48.33, 16.25 48.33, 16.25 48.41, 16.17 48.41, 16.17 48.33))")

# West of Vienna, 33UWP
# aoi = shapely.from_wkt("POLYGON ((15.00 48.75, 16.49 48.74, 16.46 47.75, 15.00 47.76, 15.00 48.75))")

aoi_gdf = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[aoi])
t_range = ["2023-06-01", "2023-07-01"] # 1 month
# t_range = ["2023-03-01", "2023-07-01"] # 4 months
epsg = 32633 # For MGRS-33UWP

# STAC-based metadata retrieval
catalog = Client.open("https://earth-search.aws.element84.com/v1")
query = catalog.search(
    collections="sentinel-2-l2a", datetime=t_range, limit=100, intersects=aoi
)
item_coll = query.item_collection()

# list results - part I
items = list(query.items())
print(f"Found: {len(items):d} items.")

# list results - part II
stac_json = query.item_collection_as_dict()
gdf = gpd.GeoDataFrame.from_features(stac_json, "epsg:4326")
gdf

Found: 13 items.


,geometry,created,platform,constellation,instruments,eo:cloud_cover,proj:epsg,mgrs:utm_zone,mgrs:latitude_band,mgrs:grid_square,...,s2:datastrip_id,s2:granule_id,s2:reflectance_conversion_factor,datetime,s2:sequence,earthsearch:s3_path,earthsearch:payload_id,earthsearch:boa_offset_applied,processing:software,updated
0,"POLYGON ((15.99815 48.74868, 16.49326 48.74333...",2023-07-01T18:36:05.617Z,sentinel-2b,sentinel-2,[msi],99.629170,32633,33,U,WP,...,S2B_OPER_MSI_L2A_DS_2BPS_20230701T113400_S2023...,S2B_OPER_MSI_L2A_TL_2BPS_20230701T113400_A0329...,0.967644,2023-07-01T09:57:27.326000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/5aec...,True,{'sentinel2-to-stac': '0.1.0'},2023-07-01T18:36:05.617Z
1,"POLYGON ((14.99974 48.75300, 16.49326 48.74333...",2023-06-29T23:46:20.169Z,sentinel-2a,sentinel-2,[msi],6.525128,32633,33,U,WP,...,S2A_OPER_MSI_L2A_DS_2APS_20230629T175255_S2023...,S2A_OPER_MSI_L2A_TL_2APS_20230629T175255_A0418...,0.967807,2023-06-29T10:07:23.808000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/d626...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-29T23:46:20.169Z
2,"POLYGON ((15.99666 48.74869, 16.49326 48.74333...",2023-06-26T17:20:57.655Z,sentinel-2a,sentinel-2,[msi],11.215745,32633,33,U,WP,...,S2A_OPER_MSI_L2A_DS_2APS_20230626T140000_S2023...,S2A_OPER_MSI_L2A_TL_2APS_20230626T140000_A0418...,0.968124,2023-06-26T09:57:25.568000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/2195...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-26T17:20:57.655Z
3,"POLYGON ((14.99974 48.75300, 16.49326 48.74333...",2023-06-24T18:11:36.093Z,sentinel-2b,sentinel-2,[msi],99.978906,32633,33,U,WP,...,S2B_OPER_MSI_L2A_DS_2BPS_20230624T113131_S2023...,S2B_OPER_MSI_L2A_TL_2BPS_20230624T113131_A0328...,0.968380,2023-06-24T10:07:23.800000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/07e0...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-24T18:11:36.093Z
4,"POLYGON ((16.00142 48.74865, 16.49326 48.74333...",2023-06-21T18:19:44.109Z,sentinel-2b,sentinel-2,[msi],30.004331,32633,33,U,WP,...,S2B_OPER_MSI_L2A_DS_2BPS_20230621T113044_S2023...,S2B_OPER_MSI_L2A_TL_2BPS_20230621T113044_A0328...,0.968835,2023-06-21T09:57:26.119000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/b647...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-21T18:19:44.109Z
5,"POLYGON ((14.99974 48.75300, 16.49326 48.74333...",2023-06-19T19:28:34.163Z,sentinel-2a,sentinel-2,[msi],37.662557,32633,33,U,WP,...,S2A_OPER_MSI_L2A_DS_2APS_20230619T160956_S2023...,S2A_OPER_MSI_L2A_TL_2APS_20230619T160956_A0417...,0.969182,2023-06-19T10:07:23.655000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/e1eb...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-19T19:28:34.163Z
6,"POLYGON ((15.99203 48.74873, 16.49326 48.74333...",2023-06-16T19:59:17.938Z,sentinel-2a,sentinel-2,[msi],16.089541,32633,33,U,WP,...,S2A_OPER_MSI_L2A_DS_2APS_20230616T153853_S2023...,S2A_OPER_MSI_L2A_TL_2APS_20230616T153853_A0416...,0.969772,2023-06-16T09:57:26.329000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/667b...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-16T19:59:17.938Z
7,"POLYGON ((14.99974 48.75300, 16.49326 48.74333...",2023-06-14T18:12:47.059Z,sentinel-2b,sentinel-2,[msi],33.078259,32633,33,U,WP,...,S2B_OPER_MSI_L2A_DS_2BPS_20230614T113003_S2023...,S2B_OPER_MSI_L2A_TL_2BPS_20230614T113003_A0327...,0.970208,2023-06-14T10:07:24.612000Z,0,s3://sentinel-cogs/sentinel-s2-l2a-cogs/33/U/W...,roda-sentinel2/workflow-sentinel2-to-stac/1ad4...,True,{'sentinel2-to-stac': '0.1.0'},2023-06-14T18:12:47.059Z
8,"POLYGON ((15.99516 48.74871, 16.49326 48.74333...",2023-06-11T18:15:25.755Z,sentinel-2b,sentinel-2,[msi],99.740505,32633,33,U,WP,...,S2B_OPER_MSI_L2A_DS_2BPS_20230611T113435_S2023...,S2B_OPER_

In [6]:
# define datacube
with open("files/layout_stac.json", "r") as file:
    dc = sq.datacube.STACCube(
        json.load(file),
        src=item_coll,
        group_by_solar_day=False,  # Pending Dask support
        trim=False,  # Pending Dask support
        dask_lazy=True,
        dask_chunk_size=2048
    )

# use same extents as for STAC query to set up the context for the datacube
space = sq.SpatialExtent(aoi_gdf.to_crs(epsg))
time = sq.TemporalExtent(*t_range)
resolution=20

2025-10-15 17:55:53,239 - semantique.processor.utils - INFO - Lazy dask computation enabled.


In [7]:
mapping = sq.mapping.Semantique()
mapping["entity"] = {}
mapping["entity"]["cloud"] = {
    "color": sq.appearance("scl").evaluate("in", [8, 9, 10])
}
mapping["entity"]["snow"] = {
    "color": sq.appearance("scl").evaluate("in", [11])
}

In [8]:
context = {
    "datacube": dc,
    "mapping": mapping,
    "space": space,
    "time": time,
    "crs": epsg,
    "tz": "UTC",
    "spatial_resolution": [-resolution, resolution],
    "cache_data": True
    # "cache_data": False
}

recipe = sq.QueryRecipe()
recipe["cloud_snow_share"] = ( 
    sq.collection(
        sq.entity("cloud"), 
        sq.entity("snow")
    )
    .merge("any")
    .reduce("percentage", "space")
)
# Additional output - leave out for now for fairer comparisons
# (reintroduce when we have better data dependency management)
# recipe["cloud_snowfree"] = ( 
#     sq.result("cloud_snow_share")
#     .evaluate("less", 10)
# )

# execute recipe
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    warnings.simplefilter("ignore", RuntimeWarning)
    response = recipe.execute(**context)

2025-10-15 17:55:53,267 - semantique.processor.core - INFO - Started parsing the semantic query
2025-10-15 17:55:53,325 - semantique.processor.core - DEBUG - Parsed the spatio-temporal extent:
<xarray.DataArray 'index' (time: 2, y: 449, x: 303)> Size: 2MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])
Coordinates:
  * time           (time) datetime64[ns] 16B 2023-06-01 2023-07-01
  * y              (y) float64 4kB 5.363e+06 5.363e+06 ... 5.354e+06 5.354e+06
  * x          

In [9]:
# Visualize task graph - to file. Can take a while!
# response["cloud_snow_share"].data.visualize(filename="data/graph_cloud_snow_share.svg")

In [10]:
# Alternative, lighter task graph explorer
response["cloud_snow_share"].data.__dask_graph__()

HighLevelGraph with 47 layers.
<dask.highlevelgraph.HighLevelGraph object at 0x7fe0681c6510>
 0. asset-table-d430f26086d18353d050b6810c2ec64b
 1. asset_table_to_reader_and_window-a478deb1b5eaa9ee27c6b02f19e7abea
 2. fetch_raster_window-0a5ef31258203b0ddbde185ea9f2c4c1
 3. getitem-5b21e533707affbd8ff8d863dd805cfb
 4. astype-27a5caa3566b401a267c4098d3731341
 5. ne-0c20b400983a18ca46a49c24220631c8
 6. where-1ea4d0552e1498b920d445cc702e0342
 7. ne-8d47136e1ec05dd7130a1f6bbb2e9dfb
 8. where-ad631554a952eb40e7a85778b72dacb1
 9. array-7582cc4aa0f7a9cf493233c2a4dd49f5
 10. where-012d1b457871ed908f86dfae7ff2a372
 11. f-a84a4d99e3fbb68755ce5dadb5094103
 12. f_0-a84a4d99e3fbb68755ce5dadb5094103
 13. transpose-b228ed4a1924c4a2c9e7fd93df58b541
 14. getitem-4ff5e35b82d12beca86ab3d2cde05904
 15. f-8f7f085e8083c2d4bdf5ab3f5ce9c631
 16. f_0-8f7f085e8083c2d4bdf5ab3f5ce9c631
 17. transpose-af9147d3fc0e17a557793bd6d2e6b921
 18. getitem-0c2a425d2cb897a3373bd476af53d557
 19. concatenate-47fe7c9b7cce4e45ab7948dbc55bb6f5
 20. _asarray_isnull-f2914599207cb7b467abca1aa823ded7
 21. where-76fd23fa20b6b65c45273ad0a58ccb78
 22. any-40899ae0be794b92fdb7b44b29af93f2
 23. any-aggregate-06500e64b5323cff380bd9d3fdd3cb2c
 24. invert-40ffe2b24ef14a7f6b1ebcd3804110ae
 25. sum-3a4edc4ce40eeb7afc08a8296b3ee5da
 26. sum-aggregate-92e1671557dba32b4b4cacdd642f9353
 27. equal-149eb22bd15bc21327c8eb44dd97e804
 28. where-9facb6c6362e96179415849e2275fde1
 29. reshape-d4dc45bbd02c34cade2c12fc106b3404
 30. isnan-8c19b454cfb3a9334151f05fbb99ab43
 31. invert-9c512d5db9536b92574a5cc1a0d36483
 32. sum-0d04510c46f2c83deeebb5a2c78afda2
 33. sum-aggregate-f9d95b2f5f93afa27ca5b22415e74348
 34. _asarray_isnull-ff57c21eb03908c85088523f90262be2
 35. where-a483c2e8b857759aad440899c5bd24f2
 36. astype-096e9adad4444606f6727d0d07c4bd38
 37. astype-bff2f6d0fbdc57637fa2eb61a67d7b94
 38. sum-8e375fe22ddc4d5441dd7066544ea9b4
 39. sum-aggregate-72fa3ecb06baad1e174ca33c3208f95c
 40. invert-e68a9c9bf801a0362997f48d02b1c623
 41. sum-775593baa171724aaab0bb511f1bd770
 42. sum-aggregate-f20de339ec6af101f100085967970a45
 43. equal-9f0174a48b38c8fb9c64cf6ce449a2ac
 44. where-36b943d3072a058cc3c655ffb56c464b
 45. divide-fa3758df75d1ae1a154d5d1490de253c
 46. multiply-b9566f3ecab806f868ff5dcea533a521

In [11]:
# Inspect the response - is it still a lazy dask array or has it been materialized?
response

{'cloud_snow_share': <xarray.DataArray 'cloud_snow_share' (time: 12)> Size: 96B
 dask.array<multiply, shape=(12,), dtype=float64, chunksize=(1,), chunktype=numpy.ndarray>
 Coordinates:
   * time          (time) datetime64[ns] 96B 2023-06-01T09:57:25.756000 ... 20...
     temporal_ref  int64 8B 0
 Attributes:
     spec:        RasterSpec(epsg=32633, bounds=(586580.0, 5353640.0, 592640.0...
     crs:         epsg:32633
     transform:   | 20.00, 0.00, 586580.00|\n| 0.00,-20.00, 5362620.00|\n| 0.0...
     resolution:  20.0
     value_type:  continuous}

In [12]:
# Run the computation!
result_cs_pc = response["cloud_snow_share"].compute()

/jupyter/semantique/semantique/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/jupyter/semantique/semantique/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/jupyter/semantique/semantique/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/jupyter/semantique/semantique/__init__.py:12: UserWarning: pkg_

In [13]:
# Compute secondary output - ignore for now.
# result_csfree_mask = response["cloud_snowfree"].compute()

In [16]:
result_cs_pc.hvplot.scatter(
    x="time", y="cloud_snow_share", marker="x", color="blue"
).opts(title="Cloud & snow percentage")

:Scatter   [time]   (cloud_snow_share)

In [15]:
# Results of secondary output - ignore for now.
# result_csfree_mask.hvplot.scatter(
#     x="time", y="cloud_snowfree", marker="x"
# ).opts(color="red", title="Cloud-free images")